In [65]:
import pandas as pd

In [66]:
# KR (full)
kr_df = pd.read_stata(
    "../data/raw/kr_data.dta",
    convert_categoricals=False
)

In [67]:


ir_chunks = pd.read_stata(
    "../data/raw/ir_data.dta",
    convert_categoricals=False,
    chunksize=100000
)

ir_cols = [
    'sdist',
    'v106', 'v190',
    'v113', 'v116', 'v119',
    'v171a', 'v157', 'v158', 'v159',
    'v463a', 'v463b'
]


ir_list = []

for chunk in ir_chunks:
    chunk = chunk[ir_cols]
    chunk = chunk.dropna()

    chunk['sanitation_score'] = (chunk['v113'] + chunk['v116'] + chunk['v119']) / 3
    chunk['education_score'] = (chunk['v106'] + chunk['v190']) / 2
    chunk['awareness_score'] = (chunk['v171a'] + chunk['v157'] + chunk['v158'] + chunk['v159']) / 4
    chunk['behavior_score'] = (chunk['v463a'] + chunk['v463b']) / 2

    ir_list.append(chunk)

ir_df = pd.concat(ir_list, ignore_index=True)

In [68]:

kr_cols = [
    'sdist',
    'hw70', 'hw71', 'hw72'
]

kr_df = kr_df[kr_cols]

In [69]:
# Replace invalid values with NaN
kr_df[['hw70', 'hw71', 'hw72']] = kr_df[['hw70', 'hw71', 'hw72']].replace(
    [9996, 9997, 9998, 9999],
    pd.NA
)

kr_df = kr_df.dropna(subset=['hw70', 'hw71', 'hw72'])

In [70]:
kr_df['health_score'] = (
    kr_df['hw70'] +
    kr_df['hw71'] +
    kr_df['hw72']
) / 3

kr_df['health_score'] = pd.to_numeric(kr_df['health_score'], errors='coerce')

kr_df = kr_df.dropna(subset=['health_score'])

In [71]:
ir_grouped = ir_df.groupby('sdist').mean().reset_index()
kr_grouped = kr_df.groupby('sdist').mean().reset_index()

final_df = pd.merge(ir_grouped, kr_grouped, on='sdist')

In [72]:
final_df.head()


,sdist,v106,v190,v113,v116,v119,v171a,v157,v158,v159,v463a,v463b,sanitation_score,education_score,awareness_score,behavior_score,hw70,hw71,hw72,health_score
0,1,1.648352,3.000000,19.461538,19.609890,1.021978,1.384615,0.456044,0.961538,1.285714,0.016484,0.0,12.895604,2.324176,1.021978,0.008242,-40.383764,-83.752768,-81.634686,-68.590406
1,2,1.682540,3.227513,13.455026,17.576720,0.989418,1.380952,0.576720,0.989418,1.153439,0.000000,0.0,10.673721,2.455026,1.025132,0.000000,-52.841463,-58.662602,-39.284553,-50.262873
2,3,1.887006,3.316384,21.163842,29.937853,1.101695,1.864407,1.000000,1.152542,1.412429,0.039548,0.0,15.472693,2.601695,1.357345,0.019774,-53.990698,-47.023256,-18.590698,-39.868217
3,4,1.838150,2.549133,25.300578,25.722543,1.028902,1.473988,0.693642,0.890173,1.161850,0.000000,0.0,16.857418,2.193642,1.054913,0.000000,-111.777311,-90.886555,-34.945378,-79.203081
4,5,1.740113,3.175141,22.666667,18.062147,1.033898,1.440678,0.536723,0.960452,1.096045,0.000000,0.0,13.438795,2.457627,1.008475,0.000000,-77.876404,-82.258427,-54.859551,-71.664794


In [73]:
final_df.shape

(707, 20)

In [ ]:
final_df.describe()

,sdist,v106,v190,v113,v116,v119,v171a,v157,v158,v159,v463a,v463b,sanitation_score,education_score,awareness_score,behavior_score,health_score
count,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000,707.000000
mean,422.250354,1.579434,2.924978,24.270725,20.192977,1.126663,1.013651,0.446312,0.179108,1.255296,0.001636,0.000480,13.161933,2.252206,0.723592,0.001058,-113.021710
std,272.351289,0.328401,0.819123,8.449763,4.986794,0.137689,0.482744,0.257762,0.160873,0.361070,0.006512,0.003991,3.079551,0.527698,0.244994,0.003789,30.833158
min,1.000000,0.687151,1.302817,12.196532,11.469799,0.695402,0.144828,0.000000,0.000000,0.242424,0.000000,0.000000,6.275253,1.078212,0.203030,0.000000,-192.685320
25%,195.500000,1.367833,2.217677,19.517711,16.479855,1.029070,0.628422,0.275186,0.080814,0.984006,0.000000,0.000000,10.971135,1.859016,0.535630,0.000000,-134.513599
50%,391.000000,1.570552,2.868966,22.798742,19.418033,1.105769,0.905028,0.397059,0.132653,1.311594,0.000000,0.000000,12.993019,2.203333,0.695364,0.000000,-116.698367
75%,595.500000,1.809276,3.495576,26.038590,23.512478,1.206919,1.339597,0.546342,0.220859,1.553823,0.000000,0.000000,14.938868,2.603544,0.885568,0.000000,-94.683709
max,932.000000,2.449153,4.825503,79.935065,41.837500,1.845161,2.589041,1.672566,1.152542,1.922156,0.100775,0.072727,27.291375,3.585616,1.513699,0.050388,7.104348


In [79]:
min_val = final_df['health_score'].min()
max_val = final_df['health_score'].max()

final_df['health_score_norm'] = (
    final_df['health_score'] - min_val
) / (max_val - min_val)

In [80]:
final_df = final_df[
    [
        'sdist',
        'sanitation_score',
        'education_score',
        'awareness_score',
        'behavior_score',
        'health_score_norm'
    ]
]
final_df.head()


,sdist,sanitation_score,education_score,awareness_score,behavior_score,health_score_norm
0,1,12.895604,2.324176,1.021978,0.008242,0.621128
1,2,10.673721,2.455026,1.025132,0.000000,0.712862
2,3,15.472693,2.601695,1.357345,0.019774,0.764890
3,4,16.857418,2.193642,1.054913,0.000000,0.568009
4,5,13.438795,2.457627,1.008475,0.000000,0.605740


In [ ]:
min_val = final_df['health_score'].min()
max_val = final_df['health_score'].max()

final_df['health_score_norm'] = (
    final_df['health_score'] - min_val
) / (max_val - min_val)

In [81]:
final_df.to_csv("../data/processed/final_district_data.csv", index=False)